# M3 Agentic AI - 郵件助理工作流

## 1. 引言

### 1.1 實驗概覽

在本模組的前面影片中，Andrew 展示了一個可以使用多種工具執行復雜任務的日曆助理示例。在本次無評分實驗中，你將探索一個類似的例子——**郵件助理智慧體**。 

該智慧體工作流可執行與郵件管理相關的多項任務，包括髮送郵件、查詢特定發件人的郵件以及刪除郵件。你將給予自然語言指令——例如“檢視老闆的未讀郵件”或“刪除 Happy Hour 的郵件”——並觀察它如何選擇合適的工具為你完成任務。

<img src="lab_overview.png" alt="日曆助理示例" width="700"/>

### 🎯 1.2 學習目標
在本實驗結束時，你將能夠使用 AISuite 將 **LLM 與工具連線**，提供自然語言指令，並觀察智慧體如何選擇、執行與驗證諸如搜尋、傳送、刪除郵件等多步任務。


## 2. 初始化環境與客戶端

與之前的實驗一樣，你現在將完成環境設定。  

In [1]:
# ================================
# 匯入
# ================================

# --- 第三方 ---
from dotenv import load_dotenv
import aisuite as ai
import json

# --- 本地 / 專案 ---
import utils
import display_functions
import email_tools



# ================================
# 環境與客戶端
# ================================
load_dotenv()          # 從 .env 載入環境變數
client = ai.Client()   # 初始化 AISuite 客戶端


## 3. 模擬郵件服務

### 3.1 元件
本實驗使用一個**模擬的郵件後端**來模擬真實世界的郵件互動。
你可以把它當作你的沙盒郵件收件箱：其中預置了訊息，方便你在無需傳送真實郵件的情況下進行練習。  

無需從零構建該後端；你將使用工具直接與之互動。其底層元件包括：

| 層級                    | 作用                             |
|-------------------------|---------------------------------|
| **FastAPI**             | 提供 REST 介面                   |
| **SQLite + SQLAlchemy** | 本地儲存並查詢郵件               |
| **Pydantic**            | 保證輸入與輸出有效性             |
| **AISuite 工具**        | 連線 LLM 與服務的橋樑            |


### 3.2 介面
該服務提供若干路由以模擬常見的郵件操作。  
接下來，這些路由將被包裝為工具，助理可自動呼叫：

- `POST /send` → 傳送新郵件  
- `GET /emails` → 列出所有郵件  
- `GET /emails/unread` → 僅顯示未讀郵件  
- `GET /emails/{id}` → 按 ID 獲取郵件  
- `GET /emails/search?q=...` → 按關鍵詞搜尋郵件  
- `GET /emails/filter` → 按收件人或日期範圍篩選  
- `PATCH /emails/{id}/read` → 將郵件標記為已讀  
- `PATCH /emails/{id}/unread` → 將郵件標記為未讀  
- `DELETE /emails/{id}` → 按 ID 刪除郵件  
- `GET /reset_database` → 重置到初始狀態（用於測試）  


> 💡 **關鍵想法：** 這些介面相當於“收件箱控制”。  
> 在接下來的步驟中，你將把它們暴露為 LLM 可呼叫的 Python 函式（工具）——將原始路由轉化為智慧體動作。

### 3.3 介面測試輔助函式
在交給 LLM 控制之前，你將先**自行測試後端**。  
`utils.test_*` 函式是針對 API 介面的快速封裝，讓你無需編寫原始 HTTP 請求即可嘗試**傳送、列表、搜尋、過濾、標記與刪除**等操作。

_**注意：** 可在頂部選單透過 File > Open 找到 utils.py 檔案_

每個輔助函式都有清晰、易懂的名稱。接下來你可以在下一個單元中取消註釋想要執行的函式，檢視其輸出，並確認郵件服務表現符合預期。  

例如，你可以測試：  
- 傳送測試郵件  
- 透過 ID 獲取郵件  
- 列出所有訊息  
- 將郵件標記為已讀或未讀  
- 刪除郵件  

在交付控制權給智慧體之前，這是你的**健全性檢查**。它讓你確信模擬的郵件服務正在執行，並表現得與真實郵件服務相同。

👉 要測試介面，請**取消註釋**你想嘗試的行，執行單元並立即檢視結果。

In [2]:
# 取消註釋你想嘗試的 'utils.test_*' 行
new_email_id = utils.test_send_email()
_ = utils.test_get_email(new_email_id['id'])
#_ = utils.test_list_emails()
#_ = utils.test_filter_emails(recipient="test@example.com")
#_ = utils.test_search_emails("lunch")
#_ = utils.test_unread_emails()
#_ = utils.test_mark_read(new_email_id['id'])
#_ = utils.test_mark_unread(new_email_id['id'])
#_ = utils.test_delete_email(new_email_id['id'])
#_ = utils.reset_database()


## 4. 郵件助理的工具層

### 4.1 為什麼需要工具？
既然介面已正常工作，下一步是將它們以 Python 函式形式暴露給 LLM，稱為**工具（tools）**。每個工具都會封裝一個 REST 路由，將原始 API 呼叫轉化為智慧體可執行的動作——如列表、讀取、搜尋、傳送、刪除或切換已讀狀態。

> 可以把工具看作智慧體的**執行器**：你給出自然語言指令（例如“檢查老闆的未讀郵件並傳送禮貌的回覆”），模型會選擇**呼叫哪些工具**以及**以什麼順序**完成任務。

### 4.2 設計提示
- 保持工具的 **docstring** 簡短、使用祈使句並聚焦具體動作。  
- 返回 **一致且緊湊的 JSON**，便於模型串聯使用結果。  
- 傾向於 **每個工具只承擔一個清晰職責**（單一路由、單一效果）。

### 4.3 可用工具
| 工具函式                          | 動作                                                                 |
|------------------------------------|------------------------------------------------------------------------|
| `list_all_emails()`                | 獲取所有郵件（按時間倒序）                                             |
| `list_unread_emails()`             | 僅檢索未讀郵件                                                         |
| `search_emails(query)`             | 按主題、正文或發件人關鍵詞搜尋                                         |
| `filter_emails(...)`               | 按收件人與/或時間範圍篩選                                              |
| `get_email(email_id)`              | 透過 ID 獲取指定郵件                                                   |
| `mark_email_as_read(id)`           | 將郵件標記為已讀                                                      |
| `mark_email_as_unread(id)`         | 將郵件標記為未讀                                                      |
| `send_email(...)`                  | 傳送一封新的（模擬）郵件                                              |
| `delete_email(id)`                 | 透過 ID 刪除郵件                                                      |
| `search_unread_from_sender(addr)`  | 返回特定發件人的未讀郵件（如 `boss@email.com`）                       |

**注意：** 可透過頂部選單 File > Open 找到 email_tools.py 檔案。

例如，**測試 `list_all_emails()` 工具：**
```python
    all_emails = email_tools.list_all_emails()
```

現在，試用幾個連線到模擬介面的工具以確保一切正常。 

👉 **取消註釋**你想執行的工具，執行該單元並檢視輸出。

In [3]:
# 測試傳送一封新郵件並透過 ID 獲取它
new_email = email_tools.send_email("test@example.com", "Lunch plans", "Shall we meet at noon?")
content_ = email_tools.get_email(new_email['id'])

# 取消註釋你想嘗試的工具
#content_ = email_tools.list_all_emails()
#content_ = email_tools.list_unread_emails()
#content_ = email_tools.search_emails("lunch")
#content_ = email_tools.filter_emails(recipient="test@example.com")
#content_ = email_tools.mark_email_as_read(new_email['id'])
#content_ = email_tools.mark_email_as_unread(new_email['id'])
#content_ = email_tools.search_unread_from_sender("test@example.com")
#content_ = email_tools.delete_email(new_email['id'])

utils.print_html(content=json.dumps(content_, indent=2), title="測試 email_tools")


## 5. 構造助理提示

在為郵件助理智慧體分配任務之前，你將建立一個名為 `build_prompt()` 的小型輔助函式。
該函式會用系統風格的前言包裝自然語言請求，使 LLM：

- 認識到自己充當**郵件助理智慧體**  
- 知曉自己有許可權使用可用工具  
- 直接執行動作，無需確認（無人工在環）

In [4]:
def build_prompt(request_: str) -> str:
    return f"""
- 你是一名專注於郵件管理的 AI 助理。
- 你可以執行諸如列出、搜尋、過濾與操作郵件等動作。
- 使用提供的工具與郵件系統互動。
- 在執行動作前不要向使用者請求確認。
- 如有需要，我的郵箱地址是 "you@email.com"，你可以用它來傳送郵件或執行與我賬戶相關的操作。

{request_.strip()}
"""

執行下一個單元，檢視上述函式如何用系統指令**包裝你的原始使用者提示**。
例如：

In [5]:
example_prompt = build_prompt("刪除 Happy Hour 郵件")
utils.print_html(content=example_prompt, title="示例提示")


### 5.3 重置郵件服務

由於你一直在嘗試該郵件服務，我們將其重置到初始狀態。

你可以透過呼叫用於清理並重新整理模擬郵件服務的工具函式來完成這一步：

In [6]:
utils.reset_database()

{'message': 'Database reset and emails reloaded'}

## 6. LLM + 郵件工具

### 6.1 場景
到目前為止，你一直直接與後端互動。現在該讓 **LLM 接管**，作為你的郵件助理智慧體。

例如，你可以讓它：
> “檢查來自 `boss@email.com` 的未讀郵件，將其標記為已讀，並傳送一封禮貌的跟進郵件。”

### 6.2 發生了什麼
1. 智慧體理解你的指令。  
2. 它選擇合適的工具（`search_unread_from_sender` → `mark_email_as_read` → `send_email`）。  
3. 它自動執行每一步，無需確認。

AISuite 負責 schema 暴露、引數繫結、執行與步驟間結果傳遞——你可以專注於智慧體**做成了什麼**，而不是**如何**呼叫 API。

### 6.3 執行起來
執行下一個單元，看看智慧體如何編排多種工具來完成你的請求。也可以用該郵件服務嘗試你自己的請求。

*你可以關注：*
- 清晰的**工具呼叫軌跡**，展示使用了哪些工具以及傳遞的引數。
- 簡潔的**最終訊息**，總結已執行的動作（例如：“找到 1 封未讀郵件，已標記為已讀，並傳送跟進郵件”）。

In [7]:
# 嘗試你自己的請求
prompt_ = build_prompt("檢查來自 boss@email.com 的未讀郵件，將其標記為已讀，並傳送一封禮貌的跟進郵件。")

response = client.chat.completions.create(
    model="openai:gpt-4.1", # LLM
    messages=[{"role": "user", "content": (
        prompt_
    )}],
    tools=[ # LLM 可訪問的工具列表
        email_tools.search_unread_from_sender,
        email_tools.list_unread_emails,
        email_tools.search_emails,
        email_tools.get_email,
        email_tools.mark_email_as_read,
        email_tools.send_email
    ],
    max_turns=5,
)

display_functions.pretty_print_chat_completion(response)



### 6.4 缺失的工具：`delete_email`

當所需工具不可用時會發生什麼？  

例如，嘗試這個請求：  
> “請刪除 alice@work.com 的郵件。”

由於 `delete_email` 工具尚未註冊，LLM 仍會嘗試響應，但無法完成刪除操作。   

<div style="background-color: #ffe4e1; padding: 12px; border-radius: 6px; color: black;">
  <h4>🔍 關鍵洞察</h4>
  <p style="margin: 0;">
    這強調了一個重要事實：<b>你提供哪些工具將直接決定智慧體能做什麼。</b>
  </p>
</div>


**可用工具**

| 工具函式                           | 動作                                                                  |
|------------------------------------|------------------------------------------------------------------------|
| `list_all_emails()`                | 獲取所有郵件（最新優先）                                              |
| `list_unread_emails()`             | 僅檢索未讀郵件                                                         |
| `search_emails(query)`             | 按主題、正文或發件人關鍵詞搜尋                                         |
| `filter_emails(...)`               | 按收件人與/或日期範圍篩選                                              |
| `get_email(email_id)`              | 按 ID 獲取指定郵件                                                     |
| `mark_email_as_read(id)`           | 將郵件標記為已讀                                                      |
| `mark_email_as_unread(id)`         | 將郵件標記為未讀                                                      |
| `send_email(...)`                  | 傳送一封新的（模擬）郵件                                              |
| `delete_email(id)`                 | 按 ID 刪除郵件                                                        |
| `search_unread_from_sender(addr)`  | 返回特定發件人的未讀郵件（如 `boss@email.com`）                       |





In [8]:
# 嘗試一個可能會呼叫不可用工具的請求
prompt_ = build_prompt("刪除 alice@work.com 的郵件")

response = client.chat.completions.create(
    model="openai:o4-mini",
    messages=[{"role": "user", "content": (
        prompt_
    )}],
    tools=[ # LLM 可訪問的工具列表
        email_tools.search_unread_from_sender,
        email_tools.list_unread_emails,
        email_tools.search_emails,
        email_tools.get_email,
        email_tools.mark_email_as_read,
        email_tools.send_email
    ],
    max_turns=5
)

display_functions.pretty_print_chat_completion(response)


#### 6.4.1 啟用 `delete_email` 後再試一次

當所需工具不可用時會發生什麼？

在上一步中，你為 LLM 提供瞭如下工具列表：
```python
    tools=[
        email_tools.search_unread_from_sender,
        email_tools.list_unread_emails,
        email_tools.search_emails,
        email_tools.get_email,
        email_tools.mark_email_as_read,
        email_tools.send_email
    ]
```

由於該列表中的可用工具有限，智慧體無法刪除郵件，因為 `delete_email` 工具尚不可用。

現在將 `delete_email` 加入工具列表並重新執行該單元。這一次，智慧體將具備完成任務所需的一切。  

> 提示：留意呼叫序列——在找到目標郵件後，智慧體應選擇 `delete_email` 來完成刪除操作。


In [9]:
prompt_ = build_prompt("刪除 alice@work.com 的郵件")

response = client.chat.completions.create(
    model="openai:o4-mini",
    messages=[{"role": "user", "content": (
        prompt_
    )}],
    tools=[
        email_tools.search_unread_from_sender,
        email_tools.list_unread_emails,
        email_tools.search_emails,
        email_tools.get_email,
        email_tools.mark_email_as_read,
        email_tools.send_email,
        email_tools.delete_email
    ],
    max_turns=5
)

display_functions.pretty_print_chat_completion(response)


### 6.5 定向操作：刪除“Happy Hour”郵件

測試收件箱預置了一封主題為 **“Happy Hour”** 的郵件。你的任務是指導智慧體定位該郵件並將其刪除。

供參考，以下是模擬資料集中該郵件的條目：

```json
{
  "id": 1,
  "sender": "eric@work.com",
  "recipient": "you@email.com",
  "subject": "Happy Hour",
  "body": "We're planning drinks this Friday!",
  "timestamp": "2025-06-13T04:48:59.096908",
  "read": false
}
````
執行下一個單元，觀察智慧體如何搜尋該郵件並將其從收件箱中刪除。

In [10]:
prompt_ = build_prompt("刪除 Happy Hour 郵件")

response = client.chat.completions.create(
    model="openai:o4-mini",
    messages=[{"role": "user", "content": (
        prompt_
    )}],
    tools=[
        email_tools.search_unread_from_sender,
        email_tools.list_unread_emails,
        email_tools.search_emails,
        email_tools.get_email,
        email_tools.mark_email_as_read,
        email_tools.send_email,
        email_tools.delete_email
    ],
    max_turns=5
)

display_functions.pretty_print_chat_completion(response)

## 7. 關鍵收穫

- 在本次無評分實驗中，**你探索了** LLM 郵件助理如何透過工具呼叫與模擬郵件服務互動。
- **工具呼叫**讓 LLM 超越文字生成——能夠呼叫函式（工具）並完成多步任務。
- 可用工具的集合決定了智慧體能與不能做的事（例如沒有 `delete_email` 就無法刪除訊息）。
- 清晰的 docstring 與一致的行為有助於 LLM 在每一步選擇合適的工具。
- AISuite 管理互動層：將 Python 函式暴露為工具、接收引數、發起 API 請求並返回結果。
- 觀察完整工作流（從提示 → 工具呼叫 → 輸出 → 最終響應）是理解並提升智慧體推理與行動的關鍵。

<div style="border:1px solid #22c55e; border-left:6px solid #16a34a; background:#dcfce7; border-radius:6px; padding:14px 16px; color:#064e3b; font-family:system-ui,-apple-system,Segoe UI,Roboto,Ubuntu,Cantarell,Noto Sans,sans-serif;">

🎉 <strong>恭喜！</strong>

你剛剛將一個由 LLM 驅動的<em>郵件助理智慧體</em>從自然語言請求引導到在模擬郵件服務上的具體行動。你看到了工具如何將純文字請求轉化為可靠的操作——列表、搜尋、標記已讀、傳送與刪除——而無需直接編寫原始 HTTP 請求。

你也瞭解了所暴露的工具定義了智慧體的真實能力：缺少某工具時智慧體能推理但不能行動；具備時則能串聯步驟、傳遞引數並準確完成你的需求。整個過程中，AISuite 負責底層管道：schema 暴露、引數繫結、執行與結果傳遞——讓你專注於“要做什麼”，而非“如何接線”。

掌握該工作流後，你已準備好設計以任務為中心的智慧體：安全透明地行動、解釋所做之事，並從簡單提示擴充套件到可靠的多步自動化，貫穿你的自有服務。🌟
</div>